# RetailPulse 360 — Data Quality & Exploratory Data Analysis

## Objective

Validate the available datasets, understand their structure and relationships, identify data-quality risks, and prepare a reliable analytical foundation for business KPI analysis.

In [21]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

In [22]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_PATH = PROJECT_ROOT / "data" / "processed"

print("Project root:", PROJECT_ROOT)
print("Raw data path:", RAW_DATA_PATH)
print("Processed data path", PROCESSED_DATA_PATH)

Project root: C:\WorkingDesktop\roadmap_projects\project_3\RetailPulse360_Portfolio_Starter\retailpulse360_portfolio_starter
Raw data path: C:\WorkingDesktop\roadmap_projects\project_3\RetailPulse360_Portfolio_Starter\retailpulse360_portfolio_starter\data\raw
Processed data path C:\WorkingDesktop\roadmap_projects\project_3\RetailPulse360_Portfolio_Starter\retailpulse360_portfolio_starter\data\processed


## Load data


In [23]:
customers = pd.read_csv(
    RAW_DATA_PATH / "customers.csv",
    parse_dates=["signup_date"]
)

products = pd.read_csv(
    RAW_DATA_PATH / "products.csv"
)

orders = pd.read_csv(
    RAW_DATA_PATH / "orders.csv",
    parse_dates=["order_date"]
)

order_items = pd.read_csv(
    RAW_DATA_PATH / "order_items.csv"
)

web_sessions = pd.read_csv(
    RAW_DATA_PATH / "web_sessions.csv",
    parse_dates=["session_date"]
)

support_tickets = pd.read_csv(
    RAW_DATA_PATH / "support_tickets.csv",
    parse_dates=["opened_date"]
)

marketing_spend = pd.read_csv(
    RAW_DATA_PATH / "marketing_spend.csv",
    parse_dates=["month"]
)

In [24]:
customer_modeling = pd.read_csv(
    PROCESSED_DATA_PATH / "customer_modeling_snapshot.csv",
    parse_dates=["snapshot_date"]
)

## Dataset inventory

In [25]:
datasets = {
    "customers": customers,
    "products": products,
    "orders": orders,
    "order_items": order_items,
    "web_sessions": web_sessions,
    "support_tickets": support_tickets,
    "marketing_spend": marketing_spend,
    "customer_modeling": customer_modeling
}

inventory = pd.DataFrame(
    {
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1]
    }
    for name, df in datasets.items()
).reset_index(drop=True)

inventory

,dataset,rows,columns
0,customers,6000,7
1,products,120,5
2,orders,30054,15
3,order_items,56183,8
4,web_sessions,76327,11
5,support_tickets,4662,8
6,marketing_spend,174,6
7,customer_modeling,6000,27


## Quality checks


In [26]:
# TODO: add analysis here
def inspect_dataset(df, name):
    print(f"\n{name.upper()}")
    print("-" * 60)
    print(f"Shape: {df.shape}")
    
    display(df.head())
    
    print("\nData Types:")
    display(
        pd.DataFrame({
            "column": df.columns,
            "dtype": df.dtypes.astype(str).values
        })
    )

In [27]:
for name, df in datasets.items():
    inspect_dataset(df, name)


CUSTOMERS
------------------------------------------------------------
Shape: (6000, 7)


,customer_id,signup_date,state,region,acquisition_channel,preferred_device,loyalty_tier
0,C000001,2025-06-14,MN,Midwest,Paid Social,Mobile,Silver
1,C000002,2025-12-11,GA,South,Paid Search,Desktop,Silver
2,C000003,2023-06-26,NY,Northeast,Organic Search,Tablet,Bronze
3,C000004,2023-10-30,IL,Midwest,Affiliate,Mobile,Bronze
4,C000005,2023-02-15,NV,West,Organic Search,Desktop,Bronze



Data Types:


,column,dtype
0,customer_id,object
1,signup_date,datetime64[ns]
2,state,object
3,region,object
4,acquisition_channel,object
5,preferred_device,object
6,loyalty_tier,object



PRODUCTS
------------------------------------------------------------
Shape: (120, 5)


...

,column,dtype
0,month,datetime64[ns]
1,channel,object
2,spend_usd,float64
3,impressions,int64
4,clicks,int64
5,leads,int64



CUSTOMER_MODELING
------------------------------------------------------------
Shape: (6000, 27)


,customer_id,snapshot_date,region,acquisition_channel,preferred_device,loyalty_tier,days_since_signup,recency_days,frequency_12m,monetary_value_12m,avg_order_value_12m,profit_12m,gross_margin_pct_12m,avg_discount_pct_12m,refund_rate_12m,category_diversity_12m,web_sessions_90d,avg_session_minutes_90d,product_views_90d,cart_adds_90d,conversion_rate_90d,cart_add_rate_90d,support_tickets_12m,avg_resolution_hours_12m,avg_csat_12m,refund_requests_12m,churned_next_90d
0,C000001,2026-05-31,Midwest,Paid Social,Mobile,Silver,351,156.00,2.00,127.09,63.55,41.71,0.33,0.00,0.00,3.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,4.00,0.00,0
1,C000002,2026-05-31,South,Paid Search,Desktop,Silver,171,201.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,0.00,2.00,6.02,5.00,0.00,0.00,0.00,0.00,0.00,4.00,0.00,1
2,C000003,2026-05-31,Northeast,Organic Search,Tablet,Bronze,1070,"1,100.00",0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,4.00,0.00,1
3,C000004,2026-05-31,Midwest,Affiliate,Mobile,Bronze,944,136.00,2.00,242.71,121.36,76.88,0.32,15.00,0.00,2.00,3.00,11.66,14.00,4.00,0.00,0.29,0.00,0.00,4.00,0.00,1
4,C000005,2026-05-31,West,Organic Search,Desktop,Bronze,1201,39.00,3.00,197.54,65.85,79.55,0.40,3.33,0.00,2.00,1.00,6.26,2.00,0.00,0.00,0.00,1.00,5.83,5.00,1.00,0



Data Types:


,column,dtype
0,customer_id,object
1,snapshot_date,datetime64[ns]
2,region,object
3,acquisition_channel,object
4,preferred_device,object
5,loyalty_tier,object
6,days_since_signup,int64
7,recency_days,float64
8,frequency_12m,float64
9,monetary_value_12m,float64


## Missing-value audit

In [28]:
def missing_value_report(df):
    report = pd.DataFrame({
        "missing_count": df.isna().sum()
    })
    
    return report.sort_values(
        "missing_count",
        ascending=False
    )


for name, df in datasets.items():
    print(f"\n{name.upper()}")
    display(missing_value_report(df))


CUSTOMERS


,missing_count
customer_id,0
signup_date,0
state,0
region,0
acquisition_channel,0
preferred_device,0
loyalty_tier,0



PRODUCTS


,missing_count
product_id,0
category,0
subcategory,0
base_price_usd,0
unit_cost_usd,0



ORDERS


...

,missing_count
ticket_id,0
customer_id,0
opened_date,0
issue_type,0
priority,0
resolution_hours,0
csat_score,0
refunded,0



MARKETING_SPEND


,missing_count
month,0
channel,0
spend_usd,0
impressions,0
clicks,0
leads,0



CUSTOMER_MODELING


,missing_count
customer_id,0
refund_rate_12m,0
refund_requests_12m,0
avg_csat_12m,0
avg_resolution_hours_12m,0
support_tickets_12m,0
cart_add_rate_90d,0
conversion_rate_90d,0
cart_adds_90d,0
product_views_90d,0


## Duplicate-row audit

In [29]:
duplicate_report = pd.DataFrame({
    "dataset": datasets.keys(),
    "duplicate_rows": [
        df.duplicated().sum()
        for df in datasets.values()
    ]
})

duplicate_report

,dataset,duplicate_rows
0,customers,0
1,products,0
2,orders,0
3,order_items,0
4,web_sessions,0
5,support_tickets,0
6,marketing_spend,0
7,customer_modeling,0


## Primary-key validation

In [32]:
primary_keys = {
    "customers": "customer_id",
    "products": "product_id",
    "orders": "order_id",
    "order_items": "order_item_id",
    "web_sessions": "session_id",
    "support_tickets": "ticket_id"
}

In [33]:
pk_results = []

for dataset_name, key in primary_keys.items():
    
    df = datasets[dataset_name]
    
    pk_results.append({
        "dataset": dataset_name,
        "primary_key": key,
        "rows": len(df),
        "unique_keys": df[key].nunique(),
        "duplicate_keys": df[key].duplicated().sum(),
        "missing_keys": df[key].isna().sum()
    })

pk_report = pd.DataFrame(pk_results)

pk_report

,dataset,primary_key,rows,unique_keys,duplicate_keys,missing_keys
0,customers,customer_id,6000,6000,0,0
1,products,product_id,120,120,0,0
2,orders,order_id,30054,30054,0,0
3,order_items,order_item_id,56183,56183,0,0
4,web_sessions,session_id,76327,76327,0,0
5,support_tickets,ticket_id,4662,4662,0,0


## Foreign-key integrity

In [34]:
def foreign_key_check(child_df, child_key, parent_df, parent_key):
    
    invalid_keys = (
        child_df.loc[
            ~child_df[child_key].isin(parent_df[parent_key]),
            child_key
        ]
        .dropna()
        .unique()
    )
    
    return len(invalid_keys)

In [35]:
fk_checks = pd.DataFrame([
    {
        "relationship": "orders → customers",
        "orphan_records": foreign_key_check(
            orders,
            "customer_id",
            customers,
            "customer_id"
        )
    },
    {
        "relationship": "order_items → orders",
        "orphan_records": foreign_key_check(
            order_items,
            "order_id",
            orders,
            "order_id"
        )
    },
    {
        "relationship": "order_items → products",
        "orphan_records": foreign_key_check(
            order_items,
            "product_id",
            products,
            "product_id"
        )
    },
    {
        "relationship": "web_sessions → customers",
        "orphan_records": foreign_key_check(
            web_sessions,
            "customer_id",
            customers,
            "customer_id"
        )
    },
    {
        "relationship": "support_tickets → customers",
        "orphan_records": foreign_key_check(
            support_tickets,
            "customer_id",
            customers,
            "customer_id"
        )
    }
])

fk_checks

,relationship,orphan_records
0,orders → customers,0
1,order_items → orders,0
2,order_items → products,0
3,web_sessions → customers,0
4,support_tickets → customers,0


## Date ranges inspect

In [36]:
date_ranges = pd.DataFrame({
    "dataset": [
        "customers",
        "orders",
        "web_sessions",
        "support_tickets",
        "marketing_spend"
    ],
    
    "min_date": [
        customers["signup_date"].min(),
        orders["order_date"].min(),
        web_sessions["session_date"].min(),
        support_tickets["opened_date"].min(),
        marketing_spend["month"].min()
    ],
    
    "max_date": [
        customers["signup_date"].max(),
        orders["order_date"].max(),
        web_sessions["session_date"].max(),
        support_tickets["opened_date"].max(),
        marketing_spend["month"].max()
    ]
})

date_ranges

,dataset,min_date,max_date
0,customers,2023-01-01,2026-05-15
1,orders,2024-01-01,2026-05-31
2,web_sessions,2024-01-01,2026-05-31
3,support_tickets,2024-01-01,2026-05-31
4,marketing_spend,2024-01-01,2026-05-01
